In [0]:
%pip install simple-salesforce pyyaml
dbutils.library.restartPython()

In [0]:
%run ./salesforce

In [0]:
import datetime
import traceback
import pandas
import warnings
from pyspark.sql import SparkSession

warnings.filterwarnings("ignore", category=FutureWarning)
spark = SparkSession.builder.getOrCreate()

In [0]:
CATALOG = "Workspace/Users/e713362@edp.pt/sf_databricks_pbi/data/current"
SCHEMA = "sf_databricks_pbi"

In [0]:
def guardar_tabela(df_pandas, nome_tabela):
    """Converts a pandas DataFrame to Spark and saves as a Delta table."""
    if df_pandas is None or df_pandas.empty:
        print(f"{nome_tabela}: sem dados, tabela não criada.")
        return

    df_spark = spark.createDataFrame(df_pandas)

    tabela_completa = f"{CATALOG}.{SCHEMA}.{nome_tabela}"

    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(tabela_completa)

    print(f"Tabela '{tabela_completa}' guardada com {df_pandas.shape[0]} registos.")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS sf_databricks_pbi")

In [0]:
def main():
    print("Conectando ao Salesforce...")

    try:
        sf = ligacao_salesforce()

        if sf is None:
            print("Erro na ligação.")
            return

        print("Ligação estabelecida com sucesso.")

        lista_ids = ids_ativos(sf)

        if not lista_ids:
            print("Sem ativos.")
            return

        print(f"Número de ativos: {len(lista_ids)}")

        periodo = [
            datetime.date.today() - datetime.timedelta(days=3000),
            datetime.date.today()
        ]

        # ── Ativos ──
        ativos = obter_ativos(sf_=sf, lista=lista_ids)
        guardar_tabela(ativos, "ativos")

        # ── Eventos ──
        eventos = obter_eventos(
            sf_=sf,
            lista=lista_ids,
            periodo=periodo,
            meio=[],
            estado=[],
            desde=None
        )

        # ── Pedidos de Operação ──
        pedidos_operacao = obter_po(
            sf_=sf,
            lista=lista_ids,
            meio=[],
            estado=[],
            desde=None
        )

        # ── Task Owner ──
        eventos = construir_task_owner(eventos, pedidos_operacao)

        guardar_tabela(eventos, "eventos")
        guardar_tabela(pedidos_operacao, "pedidos_operacao")

        # ── Empresa / Cliente ──
        empresa = obter_empresa(sf_=sf, lista=lista_ids)
        guardar_tabela(empresa, "empresa")

        # ── Controlo Técnico + Equipamentos + Tomadas ──
        ct, eq, tomadas = obter_ct_eq_tomadas(sf_=sf, lista=lista_ids)

        guardar_tabela(ct, "ct_me")
        guardar_tabela(eq, "equipamentos_me")
        guardar_tabela(tomadas, "tomadas_me")

        # ── Sintomas dos Eventos Pai ──
        if eventos is not None and not eventos.empty and "case_id" in eventos.columns:
            lista_cases = eventos["case_id"].dropna().astype(str).unique().tolist()
        elif eventos is not None and not eventos.empty and "Id" in eventos.columns:
            lista_cases = eventos["Id"].dropna().astype(str).unique().tolist()
        else:
            lista_cases = []

        if lista_cases:
            sintomas_eventos_pai = obter_sintoma_eventos_pai(
                sf_=sf,
                lista_solicitacoes=lista_cases
            )
        else:
            sintomas_eventos_pai = pandas.DataFrame()

        guardar_tabela(sintomas_eventos_pai, "sintomas_eventos_pai")

        print("Pipeline concluído com sucesso.")

    except Exception as e:
        print("Erro durante execução:")
        print(e)
        traceback.print_exc()

In [0]:
main()